# Letterboxd 리뷰 수집 · 정제 — KPop Demon Hunters

Jupyter 에서 위에서부터 셀을 순서대로 실행하세요.
**① 설치 → ② 설정 → ③ 수집 함수 → ④ 수집 실행 → ⑤ 정제 함수 → ⑥ 정제 실행** 순서입니다.

- 'OLDER' 버튼은 실제로 `<a class="next" href=".../page/N/">Older</a>` 링크입니다.
  이 노트북은 그 href 를 따라가며 버튼 클릭과 동일하게 다음 페이지로 넘어갑니다.
- 중간에 끊겨도 ④를 다시 실행하면 `data/_checkpoint.json` 으로 이어받습니다(resume).
- 워드클라우드는 만들지 않습니다. `data/tokens_all.txt` / `data/word_freq.csv` 가 추후 입력이 됩니다.


## ① 설치 (최초 1회)
`langdetect`, `nltk` 는 선택입니다(없어도 동작).

In [ ]:
# 필요 패키지 설치 (이미 있으면 건너뜀)
%pip install -q requests beautifulsoup4 lxml langdetect nltk
%pip install -q requests beautifulsoup4 lxml langdetect nltk cloudscraper

## ② 설정
목표 개수/정렬/딜레이 등은 여기서만 바꾸면 됩니다.

In [ ]:
import argparse, csv, json, os, random, re, sys, time
from dataclasses import dataclass, asdict
from typing import Optional
from collections import Counter
import requests
from bs4 import BeautifulSoup
import cloudscraper

# ---- 수집 설정 ----
FILM_SLUG   = "kpop-demon-hunters"
BASE        = "https://letterboxd.com"
SORT        = "by/activity"      # by/activity, by/added, by/length ...
TARGET      = 10_000             # 목표 리뷰 개수
DELAY_MIN, DELAY_MAX = 1.5, 3.0  # 정중한 요청 간격(초). 낮추지 말 것
MAX_RETRIES, TIMEOUT = 5, 25

OUT_DIR        = "data"
JSONL_PATH     = os.path.join(OUT_DIR, "reviews_raw.jsonl")
CSV_PATH       = os.path.join(OUT_DIR, "reviews_raw.csv")
CHECKPOINT_PATH= os.path.join(OUT_DIR, "_checkpoint.json")

# ---- 정제 설정 ----
CLEAN_PATH  = os.path.join(OUT_DIR, "cleaned_reviews.txt")
TOKENS_PATH = os.path.join(OUT_DIR, "tokens_all.txt")
FREQ_PATH   = os.path.join(OUT_DIR, "word_freq.csv")
MIN_TOKEN_LEN = 3

HEADERS = {
    "User-Agent": ("Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
                   "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36"),
    "Accept-Language": "en-US,en;q=0.9",
    "Accept": "text/html,application/xhtml+xml",
}
os.makedirs(OUT_DIR, exist_ok=True)
print("설정 완료 · 출력 폴더:", os.path.abspath(OUT_DIR))

## ③ 수집 함수 정의

In [ ]:
@dataclass
class Review:
    id: str; author: str; rating: Optional[float]; date: str; text: str; url: str

def make_session():
    # requests.Session() 대신 cloudscraper를 사용해 방화벽을 우회합니다.
    s = cloudscraper.create_scraper() 
    s.headers.update(HEADERS)
    return s

def polite_sleep():
    time.sleep(random.uniform(DELAY_MIN, DELAY_MAX))

''' def fetch(session, url):
    """GET. 429/5xx 는 지수 백오프 재시도. 영구 실패면 None."""
    backoff = 3.0
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            r = session.get(url, timeout=TIMEOUT)
            if r.status_code == 200:
                return r.text
            if r.status_code in (429, 500, 502, 503, 504):
                wait = backoff * attempt + random.uniform(0, 2)
                print(f"  [{r.status_code}] 재시도 {attempt}/{MAX_RETRIES} — {wait:.1f}s"); time.sleep(wait); continue
            if r.status_code in (403, 404):
                print(f"  [{r.status_code}] 접근 불가: {url}"); return None
            print(f"  [{r.status_code}] 예상 외 응답"); time.sleep(backoff)
        except requests.RequestException as e:
            wait = backoff * attempt
            print(f"  네트워크 오류({e}) 재시도 {attempt}/{MAX_RETRIES}"); time.sleep(wait)
    return None 
'''

#/Applications/Google\ Chrome.app/Contents/MacOS/Google\ Chrome --remote-debugging-port=9222 --user-data-dir="~/chrome_debug_profile"

from selenium import webdriver
from selenium.webdriver.chrome.options import Options

def make_session():
    # 셀레니움 모드에서는 세션 생성 대신 디버깅 크롬 드라이버를 반환 및 연결합니다.
    chrome_options = Options()
    chrome_options.add_experimental_option("debuggerAddress", "127.0.0.1:9222")
    driver = webdriver.Chrome(options=chrome_options)
    return driver

def polite_sleep():
    time.sleep(random.uniform(DELAY_MIN, DELAY_MAX))

def fetch(driver, url):
    """requests 대신 이미 켜져 있는 크롬 브라우저를 이용해 HTML을 가져옵니다."""
    try:
        driver.get(url)
        time.sleep(2) # 레터박스 페이지 로딩 대기
        return driver.page_source # 렌더링이 완료된 깨끗한 HTML 소스 반환
    except Exception as e:
        print(f"  브라우저 제어 오류: {e}")
        return None

RATING_MAP = {f"rated-{i}": i/2.0 for i in range(1, 11)}

def parse_rating(node):
    sp = node.select_one("span.rating")
    if not sp: return None
    for c in sp.get("class", []):
        if c in RATING_MAP: return RATING_MAP[c]
    return None

def parse_reviews(driver):
    """
    html 문자열 대신 driver 객체를 직접 받아서 
    현재 렌더링된 상태의 소스를 파싱합니다.
    """
    soup = BeautifulSoup(driver.page_source, "lxml")
    out = []
    
    # 1. 리뷰 리스트를 더 유연하게 탐색 (li 태그이면서 film-detail 클래스를 가진 것 위주)
    review_list = soup.select("li.film-detail, li.review")
    
    for li in review_list:
        # 리뷰 본문 탐색
        body = li.select_one(".body-text, .js-review-body, div.content")
        if not body: continue
        
        text = " ".join(p.get_text(" ", strip=True) for p in body.select("p")).strip()
        if not text: continue
        
        # 저자 및 링크 탐색
        a_name = li.select_one(".attribution .name, strong.name a, .name")
        author = a_name.get_text(strip=True) if a_name else "Unknown"
        
        # rating 추출
        rating = parse_rating(li)
        
        # URL 생성
        link = li.select_one("a.context, a[href*='/film/']")
        href = link.get("href") if link else ""
        url = BASE + href if href.startswith("/") else href
        
        rid = li.get("data-object-id") or href or f"{author}|{hash(text)&0xffffffff}"
        out.append(Review(str(rid), author, rating, "", text, url))
        
    return out


def find_older_link(html):
    """'OLDER'(a.next) 의 절대 URL. 없으면 None(마지막 페이지)."""
    a = BeautifulSoup(html, "lxml").select_one(".paginate-nextprev a.next")
    if not a or not a.get("href"): return None
    href = a["href"]; return BASE + href if href.startswith("/") else href

def load_checkpoint(start_url):
    seen, count, next_url = set(), 0, start_url
    if os.path.exists(JSONL_PATH):
        for line in open(JSONL_PATH, encoding="utf-8"):
            try: seen.add(json.loads(line)["id"]); count += 1
            except: pass
    if os.path.exists(CHECKPOINT_PATH):
        try:
            cp = json.load(open(CHECKPOINT_PATH, encoding="utf-8"))
            next_url = cp.get("next_url") or start_url
            print(f"[resume] {count}개 보유 — 이어받기")
        except: pass
    return next_url, seen, count

def save_checkpoint(next_url, count):
    json.dump({"next_url": next_url, "count": count, "ts": time.time()},
              open(CHECKPOINT_PATH, "w", encoding="utf-8"), ensure_ascii=False)

def export_csv():
    if not os.path.exists(JSONL_PATH): return
    with open(JSONL_PATH, encoding="utf-8") as f, open(CSV_PATH, "w", newline="", encoding="utf-8-sig") as out:
        w = csv.writer(out); w.writerow(["id","author","rating","date","text","url"])
        for line in f:
            try:
                r = json.loads(line); w.writerow([r["id"],r["author"],r["rating"],r["date"],r["text"],r["url"]])
            except: pass

def run_scrape(target=TARGET, sort=SORT):
    start_url = f"{BASE}/film/{FILM_SLUG}/reviews/{sort}/"
    next_url, seen, count = load_checkpoint(start_url)
    session = make_session(); jsonl = open(JSONL_PATH, "a", encoding="utf-8"); page = 0
    print(f"[start] 목표 {target}개 · {next_url}")
    try:
        while next_url and count < target:
            page += 1; print(f"[page {page}] {next_url} (누적 {count})")
            html = fetch(session, next_url)
            if html is None: break
            new = 0
            for rv in parse_reviews(session): # session이 현재는 driver입니다.
                if rv.id in seen: continue
                seen.add(rv.id); jsonl.write(json.dumps(asdict(rv), ensure_ascii=False)+"\n")
                count += 1; new += 1
                if count >= target: break
            jsonl.flush(); print(f"  +{new}")
            next_url = find_older_link(html); save_checkpoint(next_url, count)
            if new == 0 and next_url is None: print("  더 이상 페이지 없음 — 종료"); break
            polite_sleep()
    finally:
        jsonl.close(); export_csv()
    print(f"[done] 총 {count}개 → {JSONL_PATH}")
    return count
print("수집 함수 정의 완료")

## ④ 수집 실행
- **먼저 소량으로 동작 확인**(`run_scrape(target=50)`) 후, 정상이면 1만 개로.
- 1만 개는 페이지가 많아 시간이 걸립니다(딜레이 1.5~3초 × 약 800페이지). 끊겨도 다시 실행하면 이어집니다.

In [ ]:
import shutil
import os

# 기존에 꼬여버린 레터박스 데이터 폴더를 완전히 삭제합니다.
if os.path.exists("data"):
    shutil.rmtree("data")
    print("[정보] 체크포인트 및 임시 데이터를 초기화했습니다.")
else:
    print("초기화할 데이터가 없습니다.")

In [ ]:
import os
import csv
import json
import time
import random
from dataclasses import dataclass, asdict
from typing import Optional
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# ---- 환경 세팅 ----
FILM_SLUG = "kpop-demon-hunters"
BASE = "https://letterboxd.com"
SORT = "by/activity"
TARGET = 10000  

OUT_DIR = "data"
JSONL_PATH = os.path.join(OUT_DIR, "reviews_raw.jsonl")
CSV_PATH = os.path.join(OUT_DIR, "reviews_raw.csv")
CHECKPOINT_PATH = os.path.join(OUT_DIR, "_checkpoint.json")
os.makedirs(OUT_DIR, exist_ok=True)

@dataclass
class Review:
    id: str; author: str; rating: Optional[float]; date: str; text: str; url: str

def run_scrape_tag_free(target=TARGET, sort=SORT):
    start_url = f"{BASE}/film/{FILM_SLUG}/reviews/{sort}/"
    seen = set()
    count = 0
    next_url = start_url
    
    # 찌꺼기 체크포인트 방어 로직 (첫 페이지 꼬임 방지)
    if os.path.exists(JSONL_PATH):
        for line in open(JSONL_PATH, encoding="utf-8"):
            try: seen.add(json.loads(line)["id"]); count += 1
            except: pass
    if os.path.exists(CHECKPOINT_PATH):
        try:
            cp = json.load(open(CHECKPOINT_PATH, encoding="utf-8"))
            next_url = cp.get("next_url") or start_url
        except: pass

    # 디버깅 크롬 연결
    chrome_options = Options()
    chrome_options.add_experimental_option("debuggerAddress", "127.0.0.1:9222")
    driver = webdriver.Chrome(options=chrome_options)
    
    # 진짜 열려있는 레터박스 탭 매칭
    tab_found = False
    for handle in driver.window_handles:
        driver.switch_to.window(handle)
        if "letterboxd.com" in driver.current_url:
            print(f"[정보] 브라우저 탭 연결 성공: [{driver.title}]")
            tab_found = True
            break
            
    if not tab_found:
        print("[오류] 크롬 창에서 letterboxd.com 관련 탭을 찾을 수 없습니다.")
        return count

    jsonl = open(JSONL_PATH, "a", encoding="utf-8")
    page = 0
    
    print(f"[정보] 데이터 수집을 시작합니다. (목표: {target}개)")
    print("=" * 60)
    
    try:
        while next_url and count < target:
            page += 1
            print(f"[정보] [Page {page}] 브라우저 이동: {next_url}")
            
            driver.get(next_url)
            
            # 1. 태그 상관없이 클래스명에 .viewing-list가 들어간 최상위 박스 대기
            try:
                WebDriverWait(driver, 12).until(
                    EC.presence_of_element_located((By.CSS_SELECTOR, "[class*='viewing-list']"))
                )
                time.sleep(2.5) # 텍스트 안착을 위한 인간형 버퍼 분배
            except:
                print("⏳ 로딩이 지연되어 강제 파싱 단계로 진입합니다.")
            
            # 2. [핵심 수정] 태그 이름(ul, li)을 완전히 빼버리고 클래스명 구조로만 리뷰 카드들을 조준
            # 클래스명에 film-detail이나 review가 포함된 모든 요소를 유연하게 가져옵니다.
            list_items = driver.find_elements(By.CSS_SELECTOR, "[class*='viewing-list'] [class*='film-detail'], [class*='viewing-list'] > li, .film-detail")
            
            if not list_items:
                # 3. 만약 위 구조로도 실패할 경우를 대비한 최후의 방어막 (js-review 기준으로 역추적)
                list_items = driver.find_elements(By.CSS_SELECTOR, ".js-review")
                is_direct_review_mode = True
                print(f"[경고] 기본 요소 탐색 실패로 차선책(.js-review 직접 타겟팅)을 실행합니다: {len(list_items)}개 감지")
            else:
                is_direct_review_mode = False
                
            if not list_items:
                print("[오류] 리뷰 데이터를 인지하지 못했습니다.")
                break
                
            new_count = 0
            for item in list_items:
                try:
                    if is_direct_review_mode:
                        # 최후 방어막 모드일 때는 item 자체가 이미 실제 리뷰 본문입니다.
                        text = item.text.strip()
                        if not text: continue
                        author, rating, rid, url = "Unknown", None, f"rf-{hash(text)}", next_url
                    else:
                        # 일반 모드: 리뷰 카드 내부에서 본문(.js-review) 추출
                        review_elem = item.find_element(By.CLASS_NAME, "js-review")
                        text = review_elem.text.strip()
                        if not text: continue
                        
                        # 작성자 및 별점 그룹 파싱
                        author = "Unknown"
                        rating = None
                        try:
                            strip_elem = item.find_element(By.CLASS_NAME, "content-reactions-strip")
                            author = strip_elem.find_element(By.TAG_NAME, "a").text.strip()
                            
                            rating_span = strip_elem.find_element(By.CLASS_NAME, "rating")
                            classes = rating_span.get_attribute("class").split()
                            for c in classes:
                                if c.startswith("rated-"):
                                    rating = int(c.split("-")[1]) / 2.0
                        except:
                            pass

                        try:
                            href = item.find_element(By.CSS_SELECTOR, "a[href*='/film/']").get_attribute("href")
                        except:
                            href = f"/fake-url/{hash(text)}"
                        
                        url = href if href.startswith("http") else BASE + href
                        rid = item.get_attribute("data-object-id") or href

                    # 중복 적재 방지 및 저장
                    if str(rid) not in seen:
                        seen.add(str(rid))
                        rv = Review(str(rid), author, rating, "", text, url)
                        jsonl.write(json.dumps(asdict(rv), ensure_ascii=False) + "\n")
                        count += 1
                        new_count += 1
                        if count >= target: 
                            break
                except Exception:
                    continue
                    
            jsonl.flush()
            print(f"   ㄴ 이번 페이지에서 +{new_count}개 확보 완료 (누적 총합: {count}개)")
            
            # 다음 페이지 주소 링크 갱신
            try:
                next_btn = driver.find_element(By.CSS_SELECTOR, ".paginate-nextprev a.next")
                next_url = next_btn.get_attribute("href")
            except:
                next_url = None 
                
            json.dump({"next_url": next_url, "count": count, "ts": time.time()},
                      open(CHECKPOINT_PATH, "w", encoding="utf-8"), ensure_ascii=False)
            
            if new_count == 0 and next_url is None:
                print("[정보] 데이터 수집 영역의 끝에 도달했습니다.")
                break
                
            time.sleep(random.uniform(4.5, 7.0))
            
    finally:
        jsonl.close()
        if os.path.exists(JSONL_PATH):
            with open(JSONL_PATH, encoding="utf-8") as f, open(CSV_PATH, "w", newline="", encoding="utf-8-sig") as out:
                w = csv.writer(out)
                w.writerow(["id","author","rating","date","text","url"])
                for line in f:
                    try:
                        r = json.loads(line)
                        w.writerow([r["id"], r["author"], r["rating"], r["date"], r["text"], r["url"]])
                    except: pass
                    
    print(f"[완료] 최종 결과가 data/reviews_raw.csv 에 저장되었습니다.")
    return count

# 실행
run_scrape_tag_free(target=10000)

In [ ]:
# 1) 동작 확인 (소량)
run_scrape(target=50)

In [ ]:
# 2) 본 수집 — 시간이 오래 걸립니다. 끊기면 이 셀을 다시 실행(이어받기)
run_scrape(target=TARGET)

## ⑤ 정제 함수 정의
불용어/제외어는 아래 `ENGLISH_STOP`, `TITLE_STOP` 에서 자유롭게 수정하세요.

In [ ]:
# (A) 영어 기능어
ENGLISH_STOP = set("""
a an the and or but if then else when while of in on at to from by for with about against
between into through during before after above below up down out off over under again further
is are was were be been being am do does did doing have has had having will would shall should
can could may might must i me my mine myself we us our ours ourselves you your yours yourself
yourselves he him his himself she her hers herself it its itself they them their theirs themselves
this that these those what which who whom whose where why how all any both each few more most other
some such no nor not only own same so than too very s t just don now d ll m o re ve y ain aren couldn
didn doesn hadn hasn haven isn ma mightn mustn needn shan shouldn wasn weren won wouldn
as because until among also however therefore thus hence yet still even though although
get got getting gotten go goes going gone went make makes made making like likes liked one two
really actually basically literally maybe perhaps probably definitely simply pretty quite rather
much many lot lots kind sort thing things stuff way ways im ive id youre theyre dont doesnt didnt
cant couldnt wouldnt shouldnt wont isnt arent wasnt werent thats whats lets gonna wanna gotta
ok okay yeah yep nope nah lol lmao haha hahaha omg tbh imo idk etc able well good bad
watch watched watching see saw seen look looked looking feel felt feeling think thought
know knew known want wanted say said tell told give gave take took come came put
""".split())

# (B) 제목·장르 등 큰 어휘 제외 (케데헌/골든/케이팝데몬헌터스/영화/드라마/케이팝 → 영어 대응)
TITLE_STOP = set("""
kpop k-pop kpopdemonhunters kdh demon demons hunter hunters huntr huntrix
golden movie movies film films cinema cinematic drama dramas show shows
animation animated anime cartoon netflix sony soundtrack ost song songs music
korean korea
""".split())
# 캐릭터/그룹명도 빼고 싶으면 주석 해제:
# TITLE_STOP |= set("rumi mira zoey jinu sajaboys saja".split())

STOP = ENGLISH_STOP | TITLE_STOP

URL_RE   = re.compile(r"http\S+|www\.\S+")
TOKEN_RE = re.compile(r"[a-z][a-z'\-]+")

def tokenize(text):
    text = URL_RE.sub(" ", text.lower())
    out = []
    for w in TOKEN_RE.findall(text):
        w = w.strip("'-")
        if len(w) < MIN_TOKEN_LEN or w in STOP: continue
        out.append(w)
    return out

def get_lemmatizer():
    try:
        from nltk.stem import WordNetLemmatizer; import nltk
        try: nltk.data.find("corpora/wordnet")
        except LookupError: nltk.download("wordnet", quiet=True)
        lem = WordNetLemmatizer(); return lambda w: lem.lemmatize(w)
    except Exception:
        return None

def english_filter():
    try:
        from langdetect import detect, DetectorFactory; DetectorFactory.seed = 0
        def is_en(t):
            if len(t) < 20: return True
            try: return detect(t) == "en"
            except: return True
        return is_en
    except Exception:
        def is_en(t):
            if not t: return False
            return sum(c.isascii() for c in t)/len(t) > 0.9
        return is_en

def run_clean():
    if not os.path.exists(JSONL_PATH):
        raise SystemExit("먼저 ④ 수집을 실행하세요.")
    is_en = english_filter(); lemmatize = get_lemmatizer()
    if lemmatize is None: print("[info] nltk 없음 — 표제어 추출 생략")
    freq = Counter(); n_in = n_en = 0
    with open(JSONL_PATH, encoding="utf-8") as f, \
         open(CLEAN_PATH,"w",encoding="utf-8") as oc, \
         open(TOKENS_PATH,"w",encoding="utf-8") as ot:
        for line in f:
            try: rec = json.loads(line)
            except: continue
            n_in += 1; text = rec.get("text","")
            if not is_en(text): continue
            n_en += 1; toks = tokenize(text)
            if lemmatize:
                toks = [lemmatize(w) for w in toks]
                toks = [w for w in toks if w not in STOP and len(w) >= MIN_TOKEN_LEN]
            if not toks: continue
            freq.update(toks); oc.write(" ".join(toks)+"\n"); ot.write(" ".join(toks)+" ")
    with open(FREQ_PATH,"w",newline="",encoding="utf-8-sig") as f:
        w = csv.writer(f); w.writerow(["word","count"])
        for word,c in freq.most_common(): w.writerow([word,c])
    print(f"[done] 입력 {n_in} · 영어 {n_en} · 고유단어 {len(freq)}")
    print("[상위 30]"); 
    for word,c in freq.most_common(30): print(f"  {word:<16}{c}")
    return freq
print("정제 함수 정의 완료")

## ⑥ 정제 실행
`data/cleaned_reviews.txt`, `data/tokens_all.txt`, `data/word_freq.csv` 생성

In [ ]:
freq = run_clean()